In [0]:
from pyspark.sql.functions import *

In [0]:
silver_df=spark.read.table(
    "grocery_delivery_project.silver.silver_table"
)

In [0]:
display(silver_df)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

dim_customer = (
    silver_df
    .select(
        "customer_id",
        "customer_name"
    )
    .distinct()
)

dim_customer = dim_customer.withColumn(
    "customer_sk",
    row_number().over(
        Window.orderBy("customer_id")
    )
)

dim_customer = dim_customer.select(
    "customer_sk",
    "customer_id",
    "customer_name"
)

dim_customer.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable(
    "grocery_delivery_project.gold.dim_customer"
)
display(
    spark.read.table("grocery_delivery_project.gold.dim_customer")
)

In [0]:
dim_city = (
    silver_df
    .select("city")
    .distinct()
)

dim_city = dim_city.withColumn(
    "city_sk",
    row_number().over(
        Window.orderBy("city")
    )
)

dim_city = dim_city.select(
    "city_sk",
    "city"
)

dim_city.write \
.mode("overwrite") \
.saveAsTable(
    "grocery_delivery_project.gold.dim_city"
)


In [0]:
display(
    spark.read.table("grocery_delivery_project.gold.dim_city")
)

In [0]:
dim_category = (
    silver_df
    .select("category")
    .distinct()
)

dim_category = dim_category.withColumn(
    "category_sk",
    row_number().over(
        Window.orderBy("category")
    )
)

dim_category = dim_category.select(
    "category_sk",
    "category"
)

dim_category.write \
.mode("overwrite") \
.saveAsTable(
    "grocery_delivery_project.gold.dim_category"
)

In [0]:
display(
    spark.read.table("grocery_delivery_project.gold.dim_category")
)

In [0]:
from pyspark.sql.functions import *

dim_date = (
    silver_df
    .select("order_date")
    .distinct()
)

dim_date = (
    dim_date
    .withColumn(
        "date_sk",
        date_format(
            col("order_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn("day", dayofmonth("order_date"))
    .withColumn("week", weekofyear("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn(
        "month_name",
        date_format("order_date", "MMMM")
    )
    .withColumn(
        "quarter",
        quarter("order_date")
    )
    .withColumn(
        "year",
        year("order_date")
    )
)

dim_date.write \
.mode("overwrite") \
.saveAsTable(
    "grocery_delivery_project.gold.dim_date"
)


In [0]:
display(
    spark.read.table("grocery_delivery_project.gold.dim_date")
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number



dim_customer = spark.table(
    "grocery_delivery_project.gold.dim_customer"
)

dim_city = spark.table(
    "grocery_delivery_project.gold.dim_city"
)

dim_category = spark.table(
    "grocery_delivery_project.gold.dim_category"
)

dim_date = spark.table(
    "grocery_delivery_project.gold.dim_date"
)



fact_orders = (
    silver_df
    .join(
        dim_customer.select(
            "customer_sk",
            "customer_id"
        ),
        on="customer_id",
        how="left"
    )
    .join(
        dim_city.select(
            "city_sk",
            "city"
        ),
        on="city",
        how="left"
    )
    .join(
        dim_category.select(
            "category_sk",
            "category"
        ),
        on="category",
        how="left"
    )
    .join(
        dim_date.select(
            "date_sk",
            "order_date"
        ),
        on="order_date",
        how="left"
    )
)



fact_orders = fact_orders.withColumn(
    "order_sk",
    row_number().over(
        Window.orderBy("order_id")
    )
)



fact_orders = fact_orders.select(
    "order_sk",          # PK
    "order_id",          # Business Key
    "customer_sk",       # FK
    "city_sk",           # FK
    "category_sk",       # FK
    "date_sk",           # FK
    "quantity",
    "unit_price",
    "discount",
    "net_sales",
    "delivery_status",
    "batch_id",
    "source",
    "ingestion_time"
)



fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "grocery_delivery_project.gold.fact_orders"
    )

print("fact_orders table created successfully")


In [0]:
display(
    spark.read.table("grocery_delivery_project.gold.fact_orders")
)

# KPI 1: Sales by Category

In [0]:
from pyspark.sql.functions import sum

fact_df = spark.table(
    "grocery_delivery_project.gold.fact_orders"
)

dim_category = spark.table(
    "grocery_delivery_project.gold.dim_category"
)

sales_by_category = (
    fact_df
    .join(
        dim_category,
        "category_sk",
        "left"
    )
    .groupBy(
        "category"
    )
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("net_sales").alias("total_sales")
    )
)

sales_by_category.write \
.mode("overwrite") \
.saveAsTable(
    "grocery_delivery_project.gold.kpi_sales_by_category"
)

In [0]:
display(
    spark.read.table("grocery_delivery_project.gold.kpi_sales_by_category")
)

# KPI 2: Sales by City

In [0]:
from pyspark.sql.functions import sum,count

dim_city = spark.table(
    "grocery_delivery_project.gold.dim_city"
)

sales_by_city = (
    fact_df
    .join(
        dim_city,
        "city_sk",
        "left"
    )
    .groupBy(
        "city"
    )
    .agg(
        count("order_id").alias("total_orders"),
        sum("net_sales").alias("total_sales")
    )
)

sales_by_city.write \
.mode("overwrite") \
.saveAsTable(
    "grocery_delivery_project.gold.kpi_sales_by_city"
)


In [0]:
display(
    spark.read.table("grocery_delivery_project.gold.kpi_sales_by_city")
)

KPI 3: Fulfilment KPI

In [0]:
from pyspark.sql.functions import count

fulfilment_kpi = (
    fact_df
    .groupBy(
        "delivery_status"
    )
    .agg(
        count("*").alias("total_orders")
    )
)

fulfilment_kpi.write \
.mode("overwrite") \
.saveAsTable(
    "grocery_delivery_project.gold.kpi_fulfilment"
)

In [0]:
display(
    spark.read.table("grocery_delivery_project.gold.kpi_fulfilment")
)

# KPI 4: Monthly Sales Trend

In [0]:
dim_date = spark.table(
    "grocery_delivery_project.gold.dim_date"
)

monthly_sales = (
    fact_df
    .join(
        dim_date,
        "date_sk",
        "left"
    )
    .groupBy(
        "year",
        "month",
        "month_name"
    )
    .agg(
        sum("net_sales").alias("total_sales")
    )
)

monthly_sales.write \
.mode("overwrite") \
.saveAsTable(
    "grocery_delivery_project.gold.kpi_monthly_sales"
)

In [0]:
display(
    spark.table(
        "grocery_delivery_project.gold.kpi_monthly_sales"
    )
)